In [ ]:

import pandas as pd
import numpy as np
import os
RaS=30

from sklearn.model_selection import (GroupShuffleSplit,GroupKFold,GridSearchCV)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (r2_score,mean_squared_error,mean_absolute_error,mean_absolute_percentage_error)

from xgboost import XGBRegressor



file_name = "BNT-Energy2f - JPS-correct - Code2.xlsx"

train = pd.read_excel(file_name)


print("Dataset shape:", train.shape)

print("\nColumns:")
print(train.columns.tolist())



X = train.drop(columns=["Paper-ID", "Wrec (Jcm-3)"])

Y = train["Wrec (Jcm-3)"]

groups = train["Paper-ID"]




print("Number of samples:", len(train))

print("Number of ML features:", X.shape[1])

print("Target:", Y.name)

print("Number of unique publications:",groups.nunique())



gss = GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=RaS)

train_idx, test_idx = next(gss.split(X,Y,groups=groups))



X_train = X.iloc[train_idx].copy()

X_test = X.iloc[test_idx].copy()

y_train = Y.iloc[train_idx].copy()

y_test = Y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()

groups_test = groups.iloc[test_idx].copy()




common_papers = set(groups_train).intersection(set(groups_test))



print("Total samples:",len(train))

print("Training samples:",len(X_train))

print("Testing samples:",len(X_test))

print("Training percentage:",round(len(X_train) / len(train) * 100,2),"%")

print("Testing percentage:",round(len(X_test) / len(train) * 100,2),"%")

print("Training publications:",groups_train.nunique())

print("Testing publications:",groups_test.nunique())

print("Publications occurring in both sets:",common_papers)




scalerX = StandardScaler()

X_train1 = scalerX.fit_transform(X_train)

X_test1 = scalerX.transform(X_test)



scalery = StandardScaler()

y_train1 = scalery.fit_transform(y_train.values.reshape(-1, 1)).ravel()

y_test1 = scalery.transform(y_test.values.reshape(-1, 1)).ravel()


xgb_model = XGBRegressor(objective="reg:squarederror",random_state=RaS)



search_space = {"n_estimators":[100,200,300,400,500,600,800],"max_depth": [3,4,6,7,9],"gamma":[0.01],"learning_rate":[0.001,0.01,0.05,0.1,0.5,1]}



group_kfold = GroupKFold(n_splits=5)



GS = GridSearchCV(estimator=xgb_model,param_grid=search_space,scoring="neg_root_mean_squared_error",refit=True,cv=group_kfold,verbose=2,n_jobs=-1)




GS.fit(X_train1,y_train1,groups=groups_train)




print(GS.best_params_)

print("\nBest CV RMSE:")
print(-GS.best_score_)


model_xgb = GS.best_estimator_



print(model_xgb)


y_train_prediction_scaled = model_xgb.predict(X_train1)

y_test_prediction_scaled = model_xgb.predict(X_test1)



y_train_prediction = scalery.inverse_transform(y_train_prediction_scaled.reshape(-1, 1)).ravel()

y_test_prediction = scalery.inverse_transform(y_test_prediction_scaled.reshape(-1, 1)).ravel()



r2_train_initial = r2_score(y_train,y_train_prediction)

rmse_train_initial = np.sqrt(mean_squared_error(y_train,y_train_prediction))

mae_train_initial = mean_absolute_error(y_train,y_train_prediction)

mape_train_initial = (mean_absolute_percentage_error(y_train,y_train_prediction) * 100)


r2_test_initial = r2_score(y_test,y_test_prediction)

rmse_test_initial = np.sqrt(mean_squared_error(y_test,y_test_prediction))

mae_test_initial = mean_absolute_error(y_test,y_test_prediction)

mape_test_initial = (mean_absolute_percentage_error(y_test,y_test_prediction) * 100)



print("\nTRAINING")

print("R2   :", r2_train_initial)
print("RMSE :", rmse_train_initial)
print("MAE  :", mae_train_initial)
print("MAPE :", mape_train_initial, "%")


print("\nTESTING")

print("R2   :", r2_test_initial)
print("RMSE :", rmse_test_initial)
print("MAE  :", mae_test_initial)
print("MAPE :", mape_test_initial, "%")



#BOOTSTRAP


N = 1000


bootstrap_rng = np.random.default_rng(RaS)# ask



bootstrap_train_predictions = []

bootstrap_test_predictions = []






for i in range(N):

    print("Bootstrap iteration:",i + 1,"/",N)


    sample_index = bootstrap_rng.choice(np.arange(len(y_train1)),size=len(y_train1),replace=True)



    X_samples = X_train1[sample_index]

    y_samples = y_train1[sample_index]

    model_bootstrap = XGBRegressor(**GS.best_params_,objective="reg:squarederror",random_state=RaS)



    model_bootstrap.fit(X_samples,y_samples)



    test_prediction_scaled = (model_bootstrap.predict(X_test1))


    train_prediction_scaled = (model_bootstrap.predict(X_train1))


    test_prediction = scalery.inverse_transform(test_prediction_scaled.reshape(-1, 1)).ravel()


    train_prediction = scalery.inverse_transform(train_prediction_scaled.reshape(-1, 1)).ravel()


    bootstrap_test_predictions.append(test_prediction)

    bootstrap_train_predictions.append(train_prediction)

bootstrap_test_predictions = np.array(bootstrap_test_predictions)

bootstrap_train_predictions = np.array(bootstrap_train_predictions)



print("Test prediction array shape:",bootstrap_test_predictions.shape)

print("Train prediction array shape:",bootstrap_train_predictions.shape)


test_prediction_mean = np.mean(bootstrap_test_predictions,axis=0)


train_prediction_mean = np.mean(bootstrap_train_predictions,axis=0)



test_prediction_std = np.std(bootstrap_test_predictions,axis=0,ddof=1)


train_prediction_std = np.std(bootstrap_train_predictions,axis=0,ddof=1)


test_error = (y_test.values - test_prediction_mean)

train_error = (y_train.values - train_prediction_mean)



test_output = train.iloc[test_idx].copy()



test_output = test_output.rename(columns={"Wrec (Jcm-3)": "Actual_Wrec"})


test_output["Predicted_Wrec"] = (test_prediction_mean)


test_output["Error"] = (test_error)



test_output["Prediction_SD"] = (test_prediction_std)


train_output = train.iloc[train_idx].copy()


train_output = train_output.rename(columns={"Wrec (Jcm-3)": "Actual_Wrec"})


train_output["Predicted_Wrec"] = (train_prediction_mean)



train_output["Error"] = (train_error)


train_output["Prediction_SD"] = (train_prediction_std)


original_columns_without_target = [
    col for col in train.columns
    if col not in [
        "Paper-ID",
        "Wrec (Jcm-3)"
    ]
]


final_columns = (
    ["Paper-ID"]
    + original_columns_without_target
    + [
        "Actual_Wrec",
        "Predicted_Wrec",
        "Error",
        "Prediction_SD"
    ]
)


train_output = train_output[final_columns]

test_output = test_output[final_columns]


train_output_file = ("publication_based_train_bootstrap_results.xlsx")

test_output_file = ("publication_based_test_bootstrap_results.xlsx")



train_output.to_excel(train_output_file,index=False)



test_output.to_excel(test_output_file,index=False)


print("Training file:")

print(os.path.abspath(train_output_file))

print("\nTesting file:")

print(os.path.abspath(test_output_file))

print("\nTraining file exists:",os.path.exists(train_output_file))

print("Testing file exists:",os.path.exists(test_output_file))



r2_train = r2_score(y_train.values,train_prediction_mean)

rmse_train = np.sqrt(mean_squared_error(y_train.values,train_prediction_mean))

mae_train = mean_absolute_error(y_train.values,train_prediction_mean)

mape_train = (mean_absolute_percentage_error(y_train.values,train_prediction_mean) * 100)


r2_test = r2_score(y_test.values,test_prediction_mean)

rmse_test = np.sqrt(mean_squared_error(y_test.values,test_prediction_mean))

mae_test = mean_absolute_error(y_test.values,test_prediction_mean)

mape_test = (mean_absolute_percentage_error(y_test.values,test_prediction_mean) * 100)



print("FINAL BOOTSTRAP PERFORMANCE")


print("\nTRAINING DATA")

print("R2   :",r2_train)

print("RMSE :",rmse_train)

print("MAE  :",mae_train)

print("MAPE :",mape_train,"%")


print("\nTESTING DATA")

print("R2   :",r2_test)

print("RMSE :",rmse_test)

print("MAE  :",mae_test)

print("MAPE :",mape_test,"%")



